# Thu Thập Dữ Liệu PM2.5 Từ OpenAQ
### Data Collection · Ho Chi Minh City · Hourly Measurements

---

> **Pipeline tổng quát:** Tìm locations → Lấy sensors → Tải hourly measurements → Lưu CSV

---

## Mục Tiêu

| Hạng mục | Nội dung |
|---|---|
| **Bài toán** | Thu thập dữ liệu PM2.5 theo giờ tại TP.HCM từ OpenAQ API |
| **Phạm vi địa lý** | Bán kính 25km quanh trung tâm TP.HCM (10.7769°N, 106.7009°E) |
| **Nguồn dữ liệu** | OpenAQ v3 API (locations, sensors, hours endpoints) |
| **Thời gian thu thập** | Từ 01/01/2024 đến thời điểm hiện tại |
| **Output** | `openaq_hcm_pm25_from_2024_to_now.csv` — dữ liệu hourly |

---

## Cấu Trúc Notebook

| Bước | Nội dung |
|---|---|
| **1** | Cấu hình API và tham số |
| **2** | Hàm hỗ trợ (safe_get, retry logic) |
| **3** | Tìm locations PM2.5 quanh TP.HCM |
| **4** | Lấy danh sách sensors PM2.5 cho từng location |
| **5** | Tải dữ liệu hourly của từng sensor |
| **6** | Lưu và kiểm tra kết quả |

---

## Output Files

- `openaq_hcm_pm25_locations.csv` — Danh sách locations tìm được
- `openaq_hcm_pm25_sensors.csv` — Danh sách sensors PM2.5
- `openaq_hcm_pm25_from_2024_to_now.csv` — Dữ liệu hourly (main output)

---

## Lưu Ý Quan Trọng

1. **API Rate Limit**: OpenAQ có giới hạn daily request. Code có cơ chế retry 3 lần với delay 5s.
2. **Data Quality**: Nhiều sensors không có dữ liệu thực tế (found=0). Code sẽ bỏ qua các sensor này.
3. **Missing Values**: OpenAQ dùng `-999` hoặc `NaN` cho missing. Sẽ được xử lý ở bước tiền xử lý.
4. **Timezone**: Dữ liệu trả về cả UTC và local (Asia/Ho_Chi_Minh) và được chuyển đổi ở bước sau.


In [ ]:
import requests
import pandas as pd
import time
import os
from datetime import datetime, timezone

# 1) CẤU HÌNH
API_KEY = "f4434dfdc7a96726ea277773bdcbe51c5cd1d2d4986579c087d1b97867ac0000"
HEADERS = {"X-API-Key": API_KEY}

# Trung tâm TP.HCM
HCMC_LAT = 10.7769
HCMC_LON = 106.7009

# OpenAQ cho radius tối đa 25,000m
RADIUS_METERS = 25000

# PM2.5 có parameters_id = 2
PM25_PARAMETER_ID = 2

# Lấy từ 2024 đến thời điểm hiện tại
DATE_START = "2024-01-01T00:00:00Z"
DATE_END = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

OUTPUT_FILE = "openaq_hcm_pm25_from_2024_to_now.csv" # chứa dữ liệu PM2.5 theo giờ
LOCATIONS_FILE = "openaq_hcm_pm25_locations.csv" # lưu danh sách location tìm được
SENSORS_FILE = "openaq_hcm_pm25_sensors.csv" # lưu danh sách sensor PM2.5

# Xóa file cũ
for f in [OUTPUT_FILE, LOCATIONS_FILE, SENSORS_FILE]:
    if os.path.exists(f):
        os.remove(f)


# 2) HÀM HỖ TRỢ
def safe_get(url, params=None, max_retries=3, sleep_seconds=5):
    for attempt in range(max_retries):
        try:
            r = requests.get(url, headers=HEADERS, params=params, timeout=60)

            if r.status_code == 200: # Lấy dữ liệu thành công
                return r

            if r.status_code >= 500: # Thông báo lỗi sever
                print(f"[Server {r.status_code}] retry {attempt + 1}/{max_retries} ...")
                time.sleep(sleep_seconds)
                continue

            print(f"[Lỗi {r.status_code}] URL={url}") # Lỗi khác
            try:
                print(r.json())
            except Exception:
                print(r.text[:500])
            return None

        except requests.exceptions.RequestException as e: # Timeout/Mất kết nối
            print(f"[RequestException] {e} | retry {attempt + 1}/{max_retries}")
            time.sleep(sleep_seconds)

    return None

# Tìm location ở TP.HCM
def get_pm25_locations_hcm():
    """
    Tìm các location quanh TP.HCM có hỗ trợ PM2.5
    """
    all_rows = [] # danh sách chưa kết quả
    page = 1

    while True:
        url = "https://api.openaq.org/v3/locations"
        params = {
            "coordinates": f"{HCMC_LAT},{HCMC_LON}",
            "radius": RADIUS_METERS,
            "parameters_id": PM25_PARAMETER_ID,
            "limit": 1000,
            "page": page
        }

        r = safe_get(url, params=params)
        if r is None:
            break
        # Danh sách json & location
        data = r.json()
        results = data.get("results", [])

        print(f"Locations page {page}: {len(results)} dòng")

        if not results:
            break

        # Duyệt từng location (theo quốc gia)
        for loc in results:
            country_code = ((loc.get("country") or {}).get("code") or "")
            if country_code != "VN":
                continue

            all_rows.append({
                "location_id": loc.get("id"),
                "location_name": loc.get("name"),
                "locality": loc.get("locality"),
                "country_code": country_code,
                "latitude": ((loc.get("coordinates") or {}).get("latitude")),
                "longitude": ((loc.get("coordinates") or {}).get("longitude")),
                "datetime_first_utc": ((loc.get("datetimeFirst") or {}).get("utc")),
                "datetime_last_utc": ((loc.get("datetimeLast") or {}).get("utc")),
                "provider_name": ((loc.get("provider") or {}).get("name")),
                "owner_name": ((loc.get("owner") or {}).get("name")),
            })

        page += 1
        time.sleep(0.5)

    df = pd.DataFrame(all_rows).drop_duplicates(subset=["location_id"])
    return df

# Lấy sensor từ một location
def get_pm25_sensors_for_location(location_id):
    """
    Lấy sensor của 1 location, rồi lọc riêng PM2.5
    """
    url = f"https://api.openaq.org/v3/locations/{location_id}/sensors"
    r = safe_get(url)
    if r is None:
        return pd.DataFrame()

    # Danh sách json & sensor
    data = r.json()
    results = data.get("results", [])

    rows = []
    for s in results:
        parameter = s.get("parameter") or {}
        if parameter.get("id") == PM25_PARAMETER_ID or parameter.get("name") == "pm25":
            rows.append({
                "location_id": location_id,
                "sensor_id": s.get("id"),
                "parameter_id": parameter.get("id"),
                "parameter_name": parameter.get("name"),
                "parameter_units": parameter.get("units"),
                "parameter_display_name": parameter.get("displayName"),
            })

    return pd.DataFrame(rows)

# Hàm tải dữ liệu theo giờ của sensor
def fetch_sensor_hours_pm25(sensor_id, location_id, location_name, locality,
                            datetime_from, datetime_to):
    """
    Tải dữ liệu giờ của 1 sensor PM2.5 trong khoảng thời gian chỉ định
    """
    url = f"https://api.openaq.org/v3/sensors/{sensor_id}/hours"
    page = 1
    chunks = []

    while True:
        params = {
            "datetime_from": datetime_from,
            "datetime_to": datetime_to,
            "limit": 1000,
            "page": page
        }

        r = safe_get(url, params=params)
        if r is None:
            print(f"  Bỏ qua sensor {sensor_id}, trang {page}")
            break

        data = r.json()
        results = data.get("results", [])
        meta = data.get("meta", {})

        print(
            f"  sensor={sensor_id} | "
            f"page={page} | found={meta.get('found')} | rows={len(results)}"
        )

        if not results:
            break

        df_chunk = pd.json_normalize(results)

        # Chỉ giữ PM2.5
        if "parameter.name" in df_chunk.columns:
            df_chunk = df_chunk[df_chunk["parameter.name"] == "pm25"].copy()

        if df_chunk.empty:
            page += 1
            time.sleep(0.5)
            continue

        # Thông tin đã lấy cần thiết
        df_chunk_clean = pd.DataFrame({
            "location_id": location_id,
            "location_name": location_name,
            "locality": locality,
            "sensor_id": sensor_id,
            "parameter": df_chunk.get("parameter.name"),
            "value": df_chunk.get("value"),
            "unit": df_chunk.get("parameter.units"),
            "datetimeUtc": df_chunk.get("period.datetimeFrom.utc"),
            "datetimeLocal": df_chunk.get("period.datetimeFrom.local"),
            "datetimeToUtc": df_chunk.get("period.datetimeTo.utc"),
            "datetimeToLocal": df_chunk.get("period.datetimeTo.local"),
            "query_from": datetime_from,
            "query_to": datetime_to,
            "coveragePercent": df_chunk.get("coverage.percent"),
            "valid": ~df_chunk.get("value").isna() if "value" in df_chunk.columns else None,
        })

        chunks.append(df_chunk_clean)
        page += 1
        time.sleep(0.5)

    if chunks:
        return pd.concat(chunks, ignore_index=True)

    return pd.DataFrame()

# 3) LẤY LOCATION PM2.5 Ở TP.HCM

print("BƯỚC 1: Tìm location PM2.5 quanh TP.HCM...")
df_locations = get_pm25_locations_hcm()

if df_locations.empty:
    raise Exception("Không tìm thấy location PM2.5 nào gần TP.HCM.")

df_locations.to_csv(LOCATIONS_FILE, index=False, encoding="utf-8-sig")
print(f"Đã lưu {len(df_locations)} location vào {LOCATIONS_FILE}")
print(df_locations.head())


# 4) LẤY SENSOR PM2.5

print("\nBƯỚC 2: Lấy sensor PM2.5 cho từng location...")
sensor_frames = []

for _, loc in df_locations.iterrows():
    location_id = loc["location_id"]
    print(f"- location_id={location_id} | name={loc['location_name']}")
    df_s = get_pm25_sensors_for_location(location_id)

    if not df_s.empty:
        df_s["location_name"] = loc["location_name"]
        df_s["locality"] = loc["locality"]
        sensor_frames.append(df_s)

    time.sleep(0.3)

if not sensor_frames:
    raise Exception("Không lấy được sensor PM2.5 nào.")

# Gộp danh sách của toàn bộ sensor
df_sensors = pd.concat(sensor_frames, ignore_index=True).drop_duplicates(subset=["sensor_id"])
df_sensors.to_csv(SENSORS_FILE, index=False, encoding="utf-8-sig")
print(f"Đã lưu {len(df_sensors)} sensor PM2.5 vào {SENSORS_FILE}")
print(df_sensors.head())

# 5) TẢI DỮ LIỆU PM2.5 TỪ 2024 ĐẾN NAY

print(f"\nBƯỚC 3: Tải dữ liệu PM2.5 theo giờ từ {DATE_START} đến {DATE_END}...")
total_rows = 0

for _, row in df_sensors.iterrows():
    sensor_id = row["sensor_id"]
    location_id = row["location_id"]
    location_name = row["location_name"]
    locality = row["locality"]

    # Duyệt từng sensor để tải dữ liệu
    print("\n" + "=" * 80)
    print(f"Location: {location_name} | Sensor: {sensor_id} | Parameter: PM2.5")

    df_data = fetch_sensor_hours_pm25(
        sensor_id=sensor_id,
        location_id=location_id,
        location_name=location_name,
        locality=locality,
        datetime_from=DATE_START,
        datetime_to=DATE_END
    )

    if df_data.empty:
        print(f"  -> Không có dữ liệu PM2.5 cho sensor {sensor_id}")
        continue

    df_data.to_csv(
        OUTPUT_FILE,
        mode="a",
        index=False,
        header=not os.path.exists(OUTPUT_FILE),
        encoding="utf-8-sig"
    )

    total_rows += len(df_data)
    print(f"  -> Đã lưu {len(df_data)} dòng PM2.5")

print("\n" + "=" * 80)
print(f"HOÀN TẤT! Tổng số dòng PM2.5 đã lưu: {total_rows}")
print(f"File chính: {OUTPUT_FILE}")

# Xem thử 5 dòng đầu
if os.path.exists(OUTPUT_FILE):
    df_preview = pd.read_csv(OUTPUT_FILE, low_memory=False)
    print("\n5 dòng đầu của file PM2.5:")
    print(df_preview.head())



BƯỚC 1: Tìm location PM2.5 quanh TP.HCM...
Locations page 1: 11 dòng
Locations page 2: 0 dòng
Đã lưu 11 location vào openaq_hcm_pm25_locations.csv
   location_id                         location_name          locality  \
0         2446  US Diplomatic Post: Ho Chi Minh City              None   
1         7440  US Diplomatic Post: Ho Chi Minh City  Ho Chi Minh City   
2       268816                               outdoor              None   
3       268821                              outdoor2              None   
4       268929                                   od3              None   

  country_code   latitude   longitude    datetime_first_utc  \
0           VN  10.783041  106.700722  2016-02-29T01:00:00Z   
1           VN  10.782773  106.700035  2016-11-09T17:00:00Z   
2           VN  10.694527  106.721236                  None   
3           VN  10.694870  106.721544                  None   
4           VN  10.694685  106.721575                  None   

      datetime_last_utc     p

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>